In [2]:
import torch 
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

train_data=datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data=datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_loader=DataLoader(train_data,batch_size=64)
test_loader=DataLoader(test_data,batch_size=32)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten=nn.Flatten()
        self.linear_relu_stack=nn.Sequential(
            nn.Linear(28*28,512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10),
        )

    def forward(self,x):
        x=self.flatten(x)
        logits=self.linear_relu_stack(x)
        return logits

model =NeuralNetwork()

In [3]:
learning_rate=1e-3
batch_size=64
epoch=5

In [5]:
loss_fn=nn.CrossEntropyLoss()# 多输出的loss函数

In [7]:
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)# SGD随机梯度下降

In [20]:
def train_loop(dataloader,model,loss_fn,optimizer):
    size=len(dataloader.dataset)
    model.train()# 这里是开始训练模式，让model进行训练,可以随机丢弃一些数据防止model形成记忆，过饱和
    for batch,(X,y) in enumerate(dataloader):
        y_pred=model(X)
        loss=loss_fn(y_pred,y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch%100==0:
            loss,current=loss.item(),batch*batch_size+len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader,model,loss_fn):
    size=len(dataloader.dataset)
    model.eval()
    num_batches=len(dataloader)
    test_loss,correct=0,0

    with torch.no_grad():
        for i, (X, y) in enumerate(dataloader):
            y_pred = model(X)
            test_loss+=loss_fn(y_pred,y).item()
            correct+=(y_pred.argmax(1)==y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [21]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, model, loss_fn, optimizer)
    test_loop(test_loader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.322726  [   64/60000]
loss: 1.304468  [ 6464/60000]
loss: 1.144980  [12864/60000]
loss: 1.245909  [19264/60000]
loss: 1.108014  [25664/60000]
loss: 1.152096  [32064/60000]
loss: 1.157433  [38464/60000]
loss: 1.106999  [44864/60000]
loss: 1.141216  [51264/60000]
loss: 1.062150  [57664/60000]
Test Error: 
 Accuracy: 63.5%, Avg loss: 1.086284 

Epoch 2
-------------------------------
loss: 1.160143  [   64/60000]
loss: 1.159288  [ 6464/60000]
loss: 0.982233  [12864/60000]
loss: 1.112383  [19264/60000]
loss: 0.970086  [25664/60000]
loss: 1.022371  [32064/60000]
loss: 1.046995  [38464/60000]
loss: 0.997583  [44864/60000]
loss: 1.031407  [51264/60000]
loss: 0.971369  [57664/60000]
Test Error: 
 Accuracy: 64.9%, Avg loss: 0.984523 

Epoch 3
-------------------------------
loss: 1.047305  [   64/60000]
loss: 1.065920  [ 6464/60000]
loss: 0.870712  [12864/60000]
loss: 1.024239  [19264/60000]
loss: 0.883637  [25664/60000]
loss: 0.931494  [32064/600